# Regression and Classification - Error Analysis

**Course:** Introduction to Data Science  
**Dataset:** Global Weapons Systems  
**Student:** Eliav Elgazar  
**Student ID:** 324131291

## Objective

This assignment examines where regression and classification models make prediction errors and whether those errors are mainly related to the data, the model, or the way the prediction problem is defined.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    matthews_corrcoef,
    roc_auc_score,
    roc_curve,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 170)
RANDOM_STATE = 42

In [ ]:
# Load the same dataset used in the previous assignment
candidate_files = [Path("Global_Weapons_Systems.csv"), Path("Global Weapons Systems.csv")]
data_path = next(path for path in candidate_files if path.exists())
weapons = pd.read_csv(data_path)

# Rows without the regression target cannot be evaluated
weapons = weapons[weapons["Unit_Cost_USD"].notna()].reset_index(drop=True).copy()

# Regression target: log scale because unit cost spans several orders of magnitude
weapons["log10_unit_cost"] = np.log10(weapons["Unit_Cost_USD"])

# Classification target: systems in the highest cost quartile
high_cost_threshold = weapons["Unit_Cost_USD"].quantile(0.75)
weapons["is_high_cost"] = (weapons["Unit_Cost_USD"] >= high_cost_threshold).astype(int)

print("Rows used:", len(weapons))
print(f"High-cost threshold: ${high_cost_threshold:,.0f}")
print(f"Positive class share: {weapons['is_high_cost'].mean():.2%}")

## Cross-Validation Choice

All reported prediction errors are based on out-of-fold predictions from **5-fold cross-validation**. With 9,998 usable rows, five folds leave about 8,000 observations for training and about 2,000 for validation in each round. This provides a reasonable balance between stable validation results and computational cost. Regression uses shuffled `KFold`, while classification uses `StratifiedKFold` so that the high-cost class remains close to 25% in every fold.

In [ ]:
NUMERIC_FEATURES = [
    "Year_Introduced", "Effective_Range_m", "Max_Range_m", "Weight_kg", "Length_mm",
    "Barrel_Length_mm", "Muzzle_Velocity_mps", "Rate_of_Fire_rpm", "Warhead_Weight_kg",
    "Max_Speed_kmh", "Crew_Size", "Num_Operator_Nations",
]

CATEGORICAL_FEATURES = [
    "Category", "Subcategory", "Country_of_Origin", "Service_Status", "Generation",
    "Theater_of_Operation", "Export_Status", "Combat_Proven", "NATO_Compatible",
    "Operating_Environment", "Protection_Level", "Guidance_System", "Propulsion_Type",
    "Communication_System",
]

def make_preprocessor():
    numeric_pipe = Pipeline([
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=100,
            sparse_output=False,
        )),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipe, NUMERIC_FEATURES),
        ("categorical", categorical_pipe, CATEGORICAL_FEATURES),
    ])

X = weapons[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_reg = weapons["log10_unit_cost"]
y_cls = weapons["is_high_cost"]

regression_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
classification_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# 1 Regression Error Analysis

The regression target is `log10(Unit_Cost_USD)`. The log transformation prevents the most expensive systems from dominating the error scale and makes multiplicative cost differences comparable across categories. A Random Forest is used for the detailed error analysis because it is a flexible ensemble model and performs almost as well as the best regression model in cross-validation.

In [ ]:
regression_error_model = Pipeline([
    ("prep", make_preprocessor()),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_features=0.33,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])

reg_predictions = cross_val_predict(
    regression_error_model,
    X,
    y_reg,
    cv=regression_cv,
    n_jobs=1,
)

residuals = y_reg.to_numpy() - reg_predictions
absolute_errors = np.abs(residuals)

regression_analysis = weapons.copy()
regression_analysis["prediction"] = reg_predictions
regression_analysis["residual"] = residuals
regression_analysis["absolute_error"] = absolute_errors

## 1.1 Residual Analysis

Residuals are calculated as `actual - predicted`. A positive residual means the model predicted a cost that was too low, while a negative residual means the predicted cost was too high.

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.scatter(reg_predictions, residuals, alpha=0.25, s=9)
plt.axhline(0, linestyle="--")
plt.title("Residuals vs Predicted Log Unit Cost")
plt.xlabel("Predicted log10(Unit Cost)")
plt.ylabel("Residual")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4.5))
plt.hist(residuals, bins=50, edgecolor="black")
plt.axvline(0, linestyle="--")
plt.title("Distribution of Regression Residuals")
plt.xlabel("Residual")
plt.ylabel("Number of Systems")
plt.tight_layout()
plt.show()

category_error = (
    regression_analysis.groupby("Category")["absolute_error"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)

print("Mean residual:", round(residuals.mean(), 4))
print("MAE:", round(absolute_errors.mean(), 4))
print("Residual standard deviation:", round(residuals.std(ddof=1), 4))
print("\nMean absolute error by category:")
print(category_error.round(4).to_string())

### 1.1 Residual Analysis - Interpretation

The overall residual mean is about **0.006**, so there is very little average prediction bias. The MAE is about **0.314 log10 units** and the residual standard deviation is about **0.416**. The scatter does not show a clear widening pattern as predicted cost increases, so there is no strong visual evidence of heteroscedasticity on the log scale. The distribution is not symmetric, however, because a relatively small group of observations has large negative residuals. At the category level, **UAS** has the highest MAE at about **0.352**.

## 1.2 Error as a Function of Features

The next plots compare residuals and absolute errors with several central numeric characteristics and with the main categorical grouping. The purpose is to check whether prediction failures are concentrated in a specific feature region or subpopulation.

In [ ]:
features_to_plot = [
    "Weight_kg", "Length_mm", "Effective_Range_m",
    "Crew_Size", "Year_Introduced", "Num_Operator_Nations",
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feature in zip(axes.ravel(), features_to_plot):
    valid = regression_analysis[feature].notna()
    x_values = regression_analysis.loc[valid, feature]
    if feature not in {"Year_Introduced", "Num_Operator_Nations"}:
        x_values = np.log10(x_values.clip(lower=0.1))
    ax.scatter(x_values, regression_analysis.loc[valid, "residual"], s=7, alpha=0.25)
    ax.axhline(0, linestyle="--")
    ax.set_title(f"{feature} vs Residual")
    ax.set_xlabel(feature)
    ax.set_ylabel("Residual")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feature in zip(axes.ravel(), features_to_plot):
    valid = regression_analysis[feature].notna()
    x_values = regression_analysis.loc[valid, feature]
    if feature not in {"Year_Introduced", "Num_Operator_Nations"}:
        x_values = np.log10(x_values.clip(lower=0.1))
    ax.scatter(x_values, regression_analysis.loc[valid, "absolute_error"], s=7, alpha=0.25)
    ax.set_title(f"{feature} vs Absolute Error")
    ax.set_xlabel(feature)
    ax.set_ylabel("Absolute Error")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
regression_analysis.boxplot(column="absolute_error", by="Category", grid=False, rot=45)
plt.title("Absolute Error by Category")
plt.suptitle("")
plt.xlabel("Category")
plt.ylabel("Absolute Error")
plt.tight_layout()
plt.show()

### 1.2 Error as a Function of Features - Interpretation

The continuous features do not show a clear monotonic or curved error pattern. Large errors appear across much of the feature space rather than only at extreme values of weight, range, length, year, or crew size. The categorical comparison is more informative: error levels vary by system category, with UAS showing the largest average error. This suggests that the strongest subpopulation effect is category-related rather than a simple numeric threshold in one feature.

## 1.3 Analysis of Extreme Errors

The observations with the largest 5% of absolute residuals are examined separately. They are compared with systems from the same subcategory in order to distinguish unusual feature combinations from unusual target values.

In [ ]:
extreme_threshold = np.quantile(absolute_errors, 0.95)
extreme = regression_analysis[regression_analysis["absolute_error"] >= extreme_threshold].copy()

subcategory_cost_median = regression_analysis.groupby("Subcategory")["Unit_Cost_USD"].median()
extreme["cost_vs_subcategory_median"] = (
    extreme["Unit_Cost_USD"] /
    extreme["Subcategory"].map(subcategory_cost_median)
)
extreme["cost_error_factor"] = 10 ** extreme["absolute_error"]

all_cost_ratio = (
    regression_analysis["Unit_Cost_USD"] /
    regression_analysis["Subcategory"].map(subcategory_cost_median)
)
regression_analysis["abnormally_low_cost"] = all_cost_ratio < 0.15

print("Top 5% error threshold:", round(extreme_threshold, 3))
print("Number of extreme-error rows:", len(extreme))
print("Share with negative residuals:", round((extreme["residual"] < 0).mean(), 3))
print("Median true cost / subcategory median:", round(extreme["cost_vs_subcategory_median"].median(), 3))
print(
    "Share of extreme errors with cost < 15% of subcategory median:",
    round(regression_analysis.loc[extreme.index, "abnormally_low_cost"].mean(), 3),
)

show_columns = [
    "ID", "Weapon_Name", "Category", "Subcategory", "Unit_Cost_USD",
    "prediction", "residual", "absolute_error", "cost_error_factor",
    "cost_vs_subcategory_median",
]
print("\nLargest individual errors:")
print(extreme.sort_values("absolute_error", ascending=False)[show_columns].head(10).round(3).to_string(index=False))

### 1.3 Analysis of Extreme Errors - Discussion

The 95th-percentile cutoff is about **0.833 log10 units**, equivalent to a prediction error of at least roughly **6.8 times** on the original cost scale. Of the 500 observations in this group, about **98.6% are over-predictions**. The median extreme case has a recorded cost of only about **6% of the median cost of its own subcategory**, and about **97.8%** of the top-error rows fall below 15% of their subcategory median. The largest individual errors therefore look less like unusual combinations of technical features and more like a subgroup whose recorded costs are far below comparable systems. This points mainly to a data-quality or target-generation issue rather than to a failure that can be solved only by a more complex regression model.

## 1.4 Statistical Properties of Errors

In [ ]:
regression_error_statistics = pd.Series({
    "MAE": mean_absolute_error(y_reg, reg_predictions),
    "Residual standard deviation": residuals.std(ddof=1),
    "Residual skewness": stats.skew(residuals, bias=False),
    "Residual excess kurtosis": stats.kurtosis(residuals, fisher=True, bias=False),
})

print(regression_error_statistics.round(4).to_string())

### 1.4 Statistical Properties of Errors - Interpretation

For the Random Forest, MAE is about **0.314**, residual standard deviation is **0.416**, skewness is about **-1.63**, and excess kurtosis is about **3.88**. The negative skew shows that the largest errors are concentrated on the over-prediction side. The positive excess kurtosis indicates heavier tails than a normal distribution, so a small number of observations contributes disproportionately to the overall error. This is consistent with the extreme-error analysis above.

# 2 Regression Models

The required comparison uses Linear Regression, a Decision Tree Regressor, and Random Forest as the ensemble model. All three use the same features, preprocessing, and 5-fold splits.

In [ ]:
regression_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=10,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_features=0.33,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

regression_results = []
for model_name, estimator in regression_models.items():
    model_pipeline = Pipeline([
        ("prep", make_preprocessor()),
        ("model", estimator),
    ])
    scores = cross_validate(
        model_pipeline,
        X,
        y_reg,
        cv=regression_cv,
        scoring={
            "mse": "neg_mean_squared_error",
            "mae": "neg_mean_absolute_error",
            "r2": "r2",
        },
        n_jobs=1,
    )
    mse = -scores["test_mse"].mean()
    regression_results.append({
        "Model": model_name,
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "MAE": -scores["test_mae"].mean(),
        "R2": scores["test_r2"].mean(),
    })

regression_comparison = pd.DataFrame(regression_results).set_index("Model")
print(regression_comparison.round(4).to_string())

## Discussion

Linear Regression and Random Forest are effectively tied: Linear Regression reaches about **R² = 0.940, RMSE = 0.415, MAE = 0.309**, while Random Forest reaches about **R² = 0.940, RMSE = 0.416, MAE = 0.314**. The Decision Tree is weaker at about **R² = 0.929 and RMSE = 0.450**. The result shows that extra flexibility does not materially improve generalization in this dataset.

The residual plots on the log scale do not show a strong variance increase, so homoscedasticity is approximately reasonable. Normality is not satisfied because of the heavy negative tail. Linear Regression is easier to interpret and has lower variance, while a single Decision Tree can capture non-linear effects but is more sensitive to sampling variation. Random Forest reduces the variance of a single tree by averaging many trees, but here it does not outperform the linear model enough to justify a strong performance claim. The main limitation is therefore the unusual target values identified in Section 1 rather than a lack of model complexity.

# 3 Classification Error Analysis

The classification target is `is_high_cost`: 1 for systems in the top 25% of unit cost and 0 otherwise. Logistic Regression and Random Forest are evaluated with the same input features and stratified 5-fold cross-validation. Random Forest is used for the detailed error analysis because it has the slightly higher F1-score.

In [ ]:
classification_models = {
    "Logistic Regression": LogisticRegression(max_iter=3000),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_features=0.33,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

classification_probabilities = {}
classification_results = []

for model_name, estimator in classification_models.items():
    model_pipeline = Pipeline([
        ("prep", make_preprocessor()),
        ("model", estimator),
    ])
    probabilities = cross_val_predict(
        model_pipeline,
        X,
        y_cls,
        cv=classification_cv,
        method="predict_proba",
        n_jobs=1,
    )[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    classification_probabilities[model_name] = probabilities
    classification_results.append({
        "Model": model_name,
        "Precision": precision_score(y_cls, predictions, zero_division=0),
        "Recall": recall_score(y_cls, predictions, zero_division=0),
        "F1": f1_score(y_cls, predictions, zero_division=0),
        "MCC": matthews_corrcoef(y_cls, predictions),
        "ROC_AUC": roc_auc_score(y_cls, probabilities),
    })

classification_comparison = pd.DataFrame(classification_results).set_index("Model")
print(classification_comparison.round(4).to_string())

best_classifier = classification_comparison["F1"].idxmax()
best_probabilities = classification_probabilities[best_classifier]
best_predictions = (best_probabilities >= 0.5).astype(int)
print("\nClassifier used for detailed analysis:", best_classifier)

## 3.1 Confusion Matrix Analysis

In [ ]:
cm = confusion_matrix(y_cls, best_predictions)
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not High-Cost", "High-Cost"],
).plot()
plt.title(f"Confusion Matrix - {best_classifier}")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

classification_analysis = weapons[[
    "Category", "Subcategory", "Weight_kg", "Length_mm", "Effective_Range_m",
    "Max_Range_m", "Max_Speed_kmh", "Crew_Size", "Year_Introduced", "Num_Operator_Nations",
]].copy()
classification_analysis["actual"] = y_cls.to_numpy()
classification_analysis["predicted"] = best_predictions
classification_analysis["probability"] = best_probabilities
classification_analysis["is_correct"] = best_predictions == y_cls.to_numpy()

category_classification_error = (
    classification_analysis.groupby("Category")["is_correct"]
    .agg(total="count", correct_rate="mean")
)
category_classification_error["error_rate"] = 1 - category_classification_error["correct_rate"]
print("\nClassification error rate by category:")
print(category_classification_error.sort_values("error_rate", ascending=False).round(4).to_string())

### 3.1 Confusion Matrix Analysis - Interpretation

At threshold 0.5, Random Forest produces **7,206 true negatives, 292 false positives, 3 false negatives, and 2,497 true positives**. The model therefore has very high recall but still produces a noticeable number of false alarms. Errors are concentrated in three categories: **Air Defense** has the highest error rate at about **21.4%**, followed by **Aircraft** at about **8.4%** and **Naval** at about **1.7%**. The other categories have no classification errors at this threshold. In a screening context where the purpose is to identify expensive systems, false negatives are the more serious error because they represent high-cost systems that would be missed.

## 3.2 Probability-Based Analysis

In [ ]:
is_correct = best_predictions == y_cls.to_numpy()
confidence = np.where(best_predictions == 1, best_probabilities, 1 - best_probabilities)

probability_summary = pd.DataFrame({
    "Group": np.where(is_correct, "Correct", "Incorrect"),
    "Predicted probability": best_probabilities,
    "Confidence": confidence,
})

print(probability_summary.groupby("Group")[["Predicted probability", "Confidence"]].agg(["count", "mean", "median"]).round(3))

plt.figure(figsize=(8, 4.5))
plt.hist(best_probabilities[is_correct], bins=30, alpha=0.5, label="Correct")
plt.hist(best_probabilities[~is_correct], bins=30, alpha=0.5, label="Incorrect")
plt.title("Predicted High-Cost Probability: Correct vs Incorrect")
plt.xlabel("Predicted Probability")
plt.ylabel("Number of Systems")
plt.legend()
plt.tight_layout()
plt.show()

high_confidence_cutoff = 0.90
high_confidence_errors = (~is_correct) & (confidence >= high_confidence_cutoff)
print(f"High-confidence errors (confidence >= {high_confidence_cutoff:.2f}):", high_confidence_errors.sum())

### 3.2 Probability-Based Analysis - Interpretation

The model is often confident when it is correct, especially for the large group of low-cost systems. However, confidence does not eliminate mistakes. There are **68 incorrect predictions with confidence of at least 0.90**, and these are false positives concentrated in Aircraft and Naval systems. High-confidence errors are important because they show that the model can be certain even when the class boundary inside a category is not well explained by the available features.

## 3.3 Error as a Function of Features

In [ ]:
comparison_features = [
    "Weight_kg", "Length_mm", "Effective_Range_m",
    "Max_Range_m", "Max_Speed_kmh", "Crew_Size",
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feature in zip(axes.ravel(), comparison_features):
    correct_values = classification_analysis.loc[classification_analysis["is_correct"], feature].dropna()
    wrong_values = classification_analysis.loc[~classification_analysis["is_correct"], feature].dropna()

    if feature != "Crew_Size":
        correct_values = np.log10(correct_values.clip(lower=0.1))
        wrong_values = np.log10(wrong_values.clip(lower=0.1))

    ax.hist(correct_values, bins=25, alpha=0.5, density=True, label="Correct")
    ax.hist(wrong_values, bins=25, alpha=0.5, density=True, label="Incorrect")
    ax.set_title(feature)
    ax.set_ylabel("Density")
    ax.legend()

plt.tight_layout()
plt.show()

feature_comparison = []
for feature in comparison_features + ["Year_Introduced", "Num_Operator_Nations"]:
    correct_values = classification_analysis.loc[classification_analysis["is_correct"], feature].dropna()
    wrong_values = classification_analysis.loc[~classification_analysis["is_correct"], feature].dropna()
    feature_comparison.append({
        "Feature": feature,
        "Median correct": correct_values.median(),
        "Median incorrect": wrong_values.median(),
        "KS statistic": stats.ks_2samp(correct_values, wrong_values).statistic,
    })

print(pd.DataFrame(feature_comparison).set_index("Feature").sort_values("KS statistic", ascending=False).round(3).to_string())

### 3.3 Error as a Function of Features - Interpretation

Incorrect classifications are concentrated among larger and longer-range systems because those observations belong mainly to the three categories in which the high-cost threshold cuts through the category distribution. `Length_mm` shows the largest distribution difference, with a median of roughly **39,190 mm** for incorrect predictions versus **6,881 mm** for correct ones. Range and weight also differ, but these characteristics mainly identify the category rather than a clean within-category decision boundary. The feature distributions therefore explain where errors occur, but they do not reveal one numeric cutoff that would remove them.

## 3.4 Threshold Sensitivity Analysis

Precision, Recall, F1, and MCC are evaluated from thresholds 0.1 to 0.9. F-beta and the ROC curve are also examined to show the trade-off between false positives and false negatives.

In [ ]:
thresholds = np.arange(0.1, 1.0, 0.1)
threshold_rows = []
auc_value = roc_auc_score(y_cls, best_probabilities)

for threshold in thresholds:
    threshold_predictions = (best_probabilities >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(y_cls, threshold_predictions, zero_division=0),
        "Recall": recall_score(y_cls, threshold_predictions, zero_division=0),
        "F1": f1_score(y_cls, threshold_predictions, zero_division=0),
        "MCC": matthews_corrcoef(y_cls, threshold_predictions),
    })

threshold_table = pd.DataFrame(threshold_rows)
print(threshold_table.round(4).to_string(index=False))

plt.figure(figsize=(8, 4.5))
for metric in ["Precision", "Recall", "F1", "MCC"]:
    plt.plot(threshold_table["Threshold"], threshold_table[metric], marker="o", label=metric)
plt.title("Classification Metrics by Threshold")
plt.xlabel("Threshold")
plt.ylabel("Metric Value")
plt.xticks(thresholds.round(1))
plt.legend()
plt.tight_layout()
plt.show()

beta_values = np.arange(0.25, 3.25, 0.25)
f_beta_values = [
    fbeta_score(y_cls, best_predictions, beta=beta, zero_division=0)
    for beta in beta_values
]
plt.figure(figsize=(8, 4.5))
plt.plot(beta_values, f_beta_values, marker="o")
plt.title("F-beta Score as a Function of Beta")
plt.xlabel("Beta")
plt.ylabel("F-beta")
plt.tight_layout()
plt.show()

fpr, tpr, _ = roc_curve(y_cls, best_probabilities)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc_value:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()

### 3.4 Threshold Sensitivity Analysis - Interpretation

The model is stable across thresholds from about **0.1 to 0.6**. At threshold **0.2**, Recall is **1.000**, F1 is about **0.945**, and MCC is about **0.927**. Increasing the threshold beyond 0.7 sharply reduces Recall: at 0.9, Recall falls to about **0.567**. Precision improves only moderately over the same range. The ROC-AUC is about **0.989**, so the model ranks the two classes very well overall, but the threshold analysis shows that a high threshold creates many unnecessary false negatives.

## 3.5 Discussion

Random Forest has a slightly higher F1-score and MCC than Logistic Regression, but the difference is small. Their AUC values are both about 0.989, which indicates that the main class separation is already captured by relatively simple structure in the data. The main failure mode is not random classification noise across the entire dataset. Instead, errors are concentrated in Air Defense, Aircraft, and Naval, where the global cost threshold divides otherwise similar systems into two classes.

A lower threshold is preferable if the objective is to avoid missing expensive systems. However, threshold adjustment cannot fully solve the false positives because many of them occur in categories whose members overlap the global threshold. Better performance would therefore require either features that explain price variation within those categories or a classification definition that accounts for category-specific cost levels.

# 4 Final Reflection

1. **Where does the model fail most?**  
   Regression errors are largest among systems whose recorded cost is far below the median of their own subcategory. Classification errors are concentrated almost entirely in Air Defense, Aircraft, and Naval.

2. **Are the failures due to data, model, or formulation?**  
   The regression evidence points mainly to the data because the extreme observations have ordinary technical characteristics but unusually low target values. In classification, the global high-cost threshold is also an important part of the problem because it creates ambiguous labels inside three expensive categories. Model choice has a smaller effect because Linear Regression and Random Forest perform almost identically in regression, and Logistic Regression and Random Forest are also very close in classification.

3. **What improvements would you propose?**  
   Review unusually low cost values relative to their subcategory peers, add variables that explain cost differences within a category, and consider defining the high-cost class relative to category rather than with one global threshold.

4. **What insights did you gain?**  
   Aggregate performance scores can look strong while important failure patterns remain hidden. Examining the direction, location, and concentration of errors provides more useful information about whether the next improvement should focus on the model, the data, or the target definition.

**Conclusion:** The regression models already explain most of the predictable variation in log unit cost, while the largest remaining errors are concentrated in a small group of unusually low recorded costs. The classification models achieve very high overall discrimination, but their mistakes are concentrated in the categories where the global cost threshold creates overlap. In both tasks, the strongest improvements are more likely to come from better data and problem definition than from increasing model complexity.